# LeetCode 2951 — Find the Peaks bằng Python

Link đề: https://leetcode.com/problems/find-the-peaks/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findPeaks(self, mountain: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một phần tử là **đỉnh (peak)** nếu nó **lớn hơn hẳn cả hai láng giềng** (trái và phải); **phần tử đầu và cuối không bao giờ là đỉnh**; cần trả về **chỉ số** của các đỉnh.
2. Biết duyệt mảng theo **cửa sổ ba phần tử** `(trái, giữa, phải)` và vì sao vòng lặp phải chạy `i` từ `1` đến `n - 2`.
3. Biết **so sánh dây chuyền** `a < b > c` của Python, ghép bộ ba bằng `zip`, và hiểu **chỉ số âm / ngoài biên** là cạm bẫy ở hai đầu mảng.
4. Đi từ cách **hai phép so sánh tường minh** → **so sánh dây chuyền + list comprehension** → **`zip` ba lát cắt** → **dãy hiệu (tăng rồi giảm)**.
5. Tránh các lỗi kinh điển: duyệt cả hai đầu (chỉ số `-1` quấn vòng!), dùng `>=` thay vì `>` (đỉnh phẳng), chỉ so sánh với một bên, dùng `or` thay vì `and`, và trả giá trị thay vì chỉ số.

## 2. Đề bài & ví dụ

Bạn được cho một mảng `mountain` (đánh chỉ số từ `0`). Hãy trả về **mảng chỉ số** của tất cả các **đỉnh** trong `mountain`, theo **thứ tự bất kỳ**.

**Ghi chú về đề:**

- Một phần tử là **đỉnh** nếu nó **lớn hơn hẳn** (strictly greater) **các phần tử láng giềng** của nó, tức phần tử ngay trước và ngay sau.
- **Phần tử đầu tiên và cuối cùng của mảng không được xem là đỉnh.**

**Ví dụ 1**

- Input: `mountain = [2, 4, 4]`
- Output: `[]`
- Giải thích: `mountain[0]` và `mountain[2]` là phần tử đầu/cuối nên không phải đỉnh. `mountain[1] = 4` không lớn hơn hẳn `mountain[2] = 4`, nên cũng không phải đỉnh.

**Ví dụ 2**

- Input: `mountain = [1, 4, 3, 8, 5]`
- Output: `[1, 3]`
- Giải thích: `mountain[0]` và `mountain[4]` là đầu/cuối. `mountain[1] = 4 > 1` và `4 > 3` nên là đỉnh; `mountain[2] = 3` không (`3 < 4`); `mountain[3] = 8 > 3` và `8 > 5` nên là đỉnh.

**Ràng buộc:** `3 <= mountain.length <= 100`; `1 <= mountain[i] <= 100`.

In [1]:
from typing import List
import random

examples = [
    ([2, 4, 4], []),
    ([1, 4, 3, 8, 5], [1, 3]),
    ([1, 2, 3, 4, 5], []),
    ([5, 1, 5], []),
]
for mountain, expected in examples:
    print("mountain =", mountain, "-> kỳ vọng", expected)

mountain = [2, 4, 4] -> kỳ vọng []
mountain = [1, 4, 3, 8, 5] -> kỳ vọng [1, 3]
mountain = [1, 2, 3, 4, 5] -> kỳ vọng []
mountain = [5, 1, 5] -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Láng giềng của chỉ số `i`

Phần tử ở chỉ số `i` có láng giềng trái là `a[i - 1]` và láng giềng phải là `a[i + 1]`. **Phần tử đầu (`i = 0`) không có láng giềng trái; phần tử cuối (`i = n - 1`) không có láng giềng phải.** Đó là lý do đề loại hai đầu khỏi danh sách đỉnh: ta chỉ xét `i` từ `1` tới `n - 2`.

In [2]:
a = [1, 4, 3, 8, 5]
n = len(a)
i = 2
print("trái:", a[i - 1], "giữa:", a[i], "phải:", a[i + 1])    # 4 3 8
print("các i được xét:", list(range(1, n - 1)))                # [1, 2, 3]

trái: 4 giữa: 3 phải: 8
các i được xét: [1, 2, 3]


### 3.2. `range(1, n - 1)`: dừng **trước** `n - 1`

`range(a, b)` đi từ `a` đến `b - 1` (không gồm `b`). Muốn `i` chạy `1, 2, ..., n - 2` ta viết `range(1, n - 1)`. Với `n = 5` ta được `1, 2, 3`. Với `n = 3` ta được duy nhất `i = 1`.

In [3]:
for n in (3, 4, 5):
    print(n, list(range(1, n - 1)))

3 [1]
4 [1, 2]
5 [1, 2, 3]


### 3.3. So sánh dây chuyền (chained comparison)

Python cho phép viết `a < b > c`, nghĩa là **`a < b` và `b > c`** (mỗi biến ở giữa chỉ được tính một lần). Dùng cho đỉnh rất gọn: `a[i - 1] < a[i] > a[i + 1]`.

In [4]:
print(1 < 4 > 3)          # True : 1 < 4 và 4 > 3
print(1 < 4 > 4)          # False: 4 > 4 sai
print(1 < 4 > 3 > 2)      # True : cũng nối được nhiều
print(5 < 5 > 1)          # False: 5 < 5 sai

True
False
True
False


### 3.4. Chỉ số âm quấn vòng: cạm bẫy ở đầu mảng

Trong Python `a[-1]` là **phần tử cuối**. Nếu bạn lỡ xét `i = 0`, thì `a[i - 1] = a[-1]` **không báo lỗi** mà lặng lẽ lấy phần tử cuối. Đây là lỗi khó phát hiện. Còn `a[n]` (ở `i = n - 1`, `i + 1 = n`) thì báo `IndexError`.

In [5]:
a = [1, 4, 3, 8, 5]
print(a[-1])              # 5: phần tử cuối, không phải "láng giềng trái của a[0]"
try:
    print(a[len(a)])      # chỉ số n
except IndexError as e:
    print("IndexError:", e)

5
IndexError: list index out of range


### 3.5. `zip` ghép ba lát cắt và cú pháp unpack

`zip(a, b, c)` ghép phần tử thứ `t` của từng danh sách thành bộ ba, dừng theo danh sách **ngắn nhất**. Với `a[:-2]`, `a[1:-1]`, `a[2:]` ta có đúng các bộ `(trái, giữa, phải)` cho mọi `i` từ `1` đến `n - 2`.

In [6]:
a = [1, 4, 3, 8, 5]
print(a[:-2], a[1:-1], a[2:])                # [1, 4, 3] [4, 3, 8] [3, 8, 5]
for left, mid, right in zip(a[:-2], a[1:-1], a[2:]):
    print((left, mid, right), "->", left < mid > right)

[1, 4, 3] [4, 3, 8] [3, 8, 5]
(1, 4, 3) -> True
(4, 3, 8) -> False
(3, 8, 5) -> True


### 3.6. Hiệu hai phần tử liền kề

`d[t] = a[t + 1] - a[t]`: dương nghĩa là **đang tăng**, âm nghĩa là **đang giảm**, `0` nghĩa là **bằng nhau**. Một đỉnh là chỗ mà dãy hiệu chuyển từ **dương** sang **âm**.

In [7]:
a = [1, 4, 3, 8, 5]
d = [a[t + 1] - a[t] for t in range(len(a) - 1)]
print(d)                                     # [3, -1, 5, -3]
# dấu: + - + -  -> các chỗ + rồi - nằm ở phần tử a[1] và a[3]

[3, -1, 5, -3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung mảng như **dãy đồi**. Đứng ở một điểm bên trong, nhìn trái và phải: nếu cả hai bên đều **thấp hơn hẳn** thì bạn đang ở **đỉnh**. Tính tay `mountain = [1, 4, 3, 8, 5]`:

| `i` | trái `a[i-1]` | giữa `a[i]` | phải `a[i+1]` | giữa > trái? | giữa > phải? | đỉnh? |
|---|---|---|---|---|---|---|
| 1 | 1 | 4 | 3 | có | có | **có** |
| 2 | 4 | 3 | 8 | không | không | không |
| 3 | 3 | 8 | 5 | có | có | **có** |

Chỉ số `0` và `4` bị loại vì là đầu/cuối. Kết quả `[1, 3]`. Chỉ cần **một lần duyệt**, mỗi `i` kiểm tra hai điều kiện, cả hai phải đúng.

## 5. Cách 1 — Brute Force / baseline: hai phép so sánh tường minh

**Ý tưởng.** Với mỗi `i` từ `1` đến `n - 2`, kiểm tra `a[i] > a[i - 1]` **và** `a[i] > a[i + 1]`; nếu cả hai đúng thì `append(i)`. Viết rõ từng phép so sánh để thấy đúng định nghĩa.

In [8]:
def find_peaks_explicit(mountain: List[int]) -> List[int]:
    ans = []
    n = len(mountain)
    for i in range(1, n - 1):                       # bỏ đầu và cuối
        bigger_than_left = mountain[i] > mountain[i - 1]
        bigger_than_right = mountain[i] > mountain[i + 1]
        if bigger_than_left and bigger_than_right:
            ans.append(i)
    return ans


for mountain, expected in examples:
    print(mountain, "->", find_peaks_explicit(mountain), "(kỳ vọng", expected, ")")

[2, 4, 4] -> [] (kỳ vọng [] )
[1, 4, 3, 8, 5] -> [1, 3] (kỳ vọng [1, 3] )
[1, 2, 3, 4, 5] -> [] (kỳ vọng [] )
[5, 1, 5] -> [] (kỳ vọng [] )


In [9]:
# Từng bước với ví dụ 2: in hai điều kiện
a = [1, 4, 3, 8, 5]
for i in range(1, len(a) - 1):
    left_ok, right_ok = a[i] > a[i - 1], a[i] > a[i + 1]
    print(f"i={i}: {a[i - 1]} < {a[i]} > {a[i + 1]} -> trái:{left_ok!s:5} phải:{right_ok!s:5} đỉnh:{left_ok and right_ok}")

i=1: 1 < 4 > 3 -> trái:True  phải:True  đỉnh:True
i=2: 4 < 3 > 8 -> trái:False phải:False đỉnh:False
i=3: 3 < 8 > 5 -> trái:True  phải:True  đỉnh:True


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài mảng. Mỗi phần tử bên trong được xét một lần với hai phép so sánh: **Time `O(n)`**, ngoài mảng kết quả chỉ dùng vài biến: **Space `O(1)`** phụ.

**Nút thắt.** Đã tối ưu về bậc (không thể nhanh hơn `O(n)` vì phải nhìn mọi phần tử). Chỉ còn chuyện **viết gọn và khó sai** hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Bài này không có "bước đột phá" về độ phức tạp; điểm mấu chốt là **chọn đúng phạm vi duyệt và đúng điều kiện**:

1. Phạm vi: `i` từ `1` đến `n - 2`, để **không bao giờ chạm biên** (đỡ lo `IndexError` và chỉ số âm quấn vòng).
2. Điều kiện: **lớn hơn hẳn cả hai bên**, tức `a[i - 1] < a[i] > a[i + 1]`, một biểu thức dây chuyền duy nhất.
3. Cách khác nhìn bài toán: đỉnh là chỗ dãy **tăng rồi giảm**, tức dãy hiệu đổi dấu từ dương sang âm.

Các cách bên dưới chỉ là bốn cách **viết** cùng ý đó.

## 7. Các cách tiếp theo

### Cách 2 — So sánh dây chuyền + list comprehension

**Giải thích.** Gộp hai phép so sánh bằng `a[i - 1] < a[i] > a[i + 1]` và gom vào list comprehension: "lấy `i` với mọi `i` trong `range(1, n - 1)` mà điều kiện đúng".

In [10]:
def find_peaks_chain(mountain: List[int]) -> List[int]:
    a = mountain
    return [i for i in range(1, len(a) - 1) if a[i - 1] < a[i] > a[i + 1]]


for mountain, expected in examples:
    print(mountain, "->", find_peaks_chain(mountain), "(kỳ vọng", expected, ")")

[2, 4, 4] -> [] (kỳ vọng [] )
[1, 4, 3, 8, 5] -> [1, 3] (kỳ vọng [1, 3] )
[1, 2, 3, 4, 5] -> [] (kỳ vọng [] )
[5, 1, 5] -> [] (kỳ vọng [] )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** phụ (mảng kết quả là đầu ra).

### Cách 3 — `zip` ba lát cắt

**Giải thích.** Ghép `a[:-2]`, `a[1:-1]`, `a[2:]` thành các bộ `(trái, giữa, phải)`. Bộ thứ `t` ứng với chỉ số `i = t + 1`. Dùng `enumerate(..., start=1)` để có luôn `i`. Cách này không cần viết `i - 1`, `i + 1` nên khó nhầm chỉ số, nhưng **tạo ba lát cắt** nên tốn `O(n)` bộ nhớ.

In [11]:
def find_peaks_zip(mountain: List[int]) -> List[int]:
    a = mountain
    return [
        i
        for i, (left, mid, right) in enumerate(zip(a[:-2], a[1:-1], a[2:]), start=1)
        if left < mid > right
    ]


for mountain, expected in examples:
    print(mountain, "->", find_peaks_zip(mountain), "(kỳ vọng", expected, ")")

# Kiểm tra start=1: bộ thứ t ứng với i = t + 1
a = [1, 4, 3, 8, 5]
print(list(enumerate(zip(a[:-2], a[1:-1], a[2:]), start=1)))

[2, 4, 4] -> [] (kỳ vọng [] )
[1, 4, 3, 8, 5] -> [1, 3] (kỳ vọng [1, 3] )
[1, 2, 3, 4, 5] -> [] (kỳ vọng [] )
[5, 1, 5] -> [] (kỳ vọng [] )
[(1, (1, 4, 3)), (2, (4, 3, 8)), (3, (3, 8, 5))]


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** do ba lát cắt.

### Cách 4 — Dãy hiệu: dấu `+` rồi dấu `-`

**Giải thích.** Tính `d[t] = a[t + 1] - a[t]`. Chỗ `d[t - 1] > 0` (đang lên) và `d[t] < 0` (xuống tiếp) thì `a[t]` là đỉnh. Cách này **tách việc** "tính hướng đi" khỏi việc "tìm đỉnh", và mở rộng được cho bài có **đỉnh phẳng** (ví dụ `[1, 3, 3, 2]`).

In [12]:
def find_peaks_diff(mountain: List[int]) -> List[int]:
    a = mountain
    d = [a[t + 1] - a[t] for t in range(len(a) - 1)]      # d[t]: từ a[t] sang a[t+1]
    ans = []
    for t in range(1, len(a) - 1):
        if d[t - 1] > 0 and d[t] < 0:                     # lên rồi xuống tại a[t]
            ans.append(t)
    return ans


for mountain, expected in examples:
    print(mountain, "->", find_peaks_diff(mountain), "(kỳ vọng", expected, ")")

[2, 4, 4] -> [] (kỳ vọng [] )
[1, 4, 3, 8, 5] -> [1, 3] (kỳ vọng [1, 3] )
[1, 2, 3, 4, 5] -> [] (kỳ vọng [] )
[5, 1, 5] -> [] (kỳ vọng [] )


In [13]:
# Code nộp LeetCode cho Cách 2 (gọn, đúng, O(1) bộ nhớ phụ)
from typing import List

class Solution:
    def findPeaks(self, mountain: List[int]) -> List[int]:
        return [i for i in range(1, len(mountain) - 1)
                if mountain[i - 1] < mountain[i] > mountain[i + 1]]

**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** cho mảng hiệu (có thể tính "tại chỗ" để còn `O(1)`).

## 8. TRACE từng bước

Ta theo dõi (a) Cách 1 trên `[1, 4, 3, 8, 5]`; (b) Cách 4 qua dãy hiệu.

In [14]:
def trace_explicit(mountain):
    ans = []
    n = len(mountain)
    print("mountain =", mountain, "| n =", n, "| xét i =", list(range(1, n - 1)))
    for i in range(1, n - 1):
        l, m, r = mountain[i - 1], mountain[i], mountain[i + 1]
        lt, gt = l < m, m > r
        is_peak = lt and gt
        if is_peak:
            ans.append(i)
        print(f"i={i}: ({l}, {m}, {r}) {l}<{m}? {lt!s:5} {m}>{r}? {gt!s:5} -> {'ĐỈNH' if is_peak else 'không'} | ans={ans}")
    print("=> kết quả:", ans)
    return ans


assert trace_explicit([1, 4, 3, 8, 5]) == [1, 3]
assert trace_explicit([2, 4, 4]) == []

mountain = [1, 4, 3, 8, 5] | n = 5 | xét i = [1, 2, 3]
i=1: (1, 4, 3) 1<4? True  4>3? True  -> ĐỈNH | ans=[1]
i=2: (4, 3, 8) 4<3? False 3>8? False -> không | ans=[1]
i=3: (3, 8, 5) 3<8? True  8>5? True  -> ĐỈNH | ans=[1, 3]
=> kết quả: [1, 3]
mountain = [2, 4, 4] | n = 3 | xét i = [1]
i=1: (2, 4, 4) 2<4? True  4>4? False -> không | ans=[]
=> kết quả: []


In [15]:
def trace_diff(mountain):
    a = mountain
    d = [a[t + 1] - a[t] for t in range(len(a) - 1)]
    signs = ["+" if x > 0 else "-" if x < 0 else "0" for x in d]
    print("a    =", a)
    print("d    =", d)
    print("dấu  =", signs)
    ans = []
    for t in range(1, len(a) - 1):
        if d[t - 1] > 0 and d[t] < 0:
            ans.append(t)
            print(f"t={t}: dấu trước '{signs[t - 1]}' và dấu sau '{signs[t]}' -> đỉnh a[{t}]={a[t]}")
    print("=> kết quả:", ans)
    return ans


assert trace_diff([1, 4, 3, 8, 5]) == [1, 3]

a    = [1, 4, 3, 8, 5]
d    = [3, -1, 5, -3]
dấu  = ['+', '-', '+', '-']
t=1: dấu trước '+' và dấu sau '-' -> đỉnh a[1]=4
t=3: dấu trước '+' và dấu sau '-' -> đỉnh a[3]=8
=> kết quả: [1, 3]


Bảng tóm tắt (Cách 1, `[1, 4, 3, 8, 5]`):

| `i` | `(trái, giữa, phải)` | trái < giữa | giữa > phải | `ans` |
|---|---|---|---|---|
| 1 | `(1, 4, 3)` | True | True | `[1]` |
| 2 | `(4, 3, 8)` | False | False | `[1]` |
| 3 | `(3, 8, 5)` | True | True | `[1, 3]` |

## 9. Code nộp LeetCode

Bản nộp dùng so sánh dây chuyền trong vòng `for`. Đúng class `Solution`, đúng method `findPeaks(self, mountain: List[int]) -> List[int]`.

In [16]:
from typing import List

class Solution:
    def findPeaks(self, mountain: List[int]) -> List[int]:
        ans = []
        for i in range(1, len(mountain) - 1):                  # bỏ phần tử đầu và cuối
            if mountain[i - 1] < mountain[i] > mountain[i + 1]:  # lớn hơn hẳn cả hai bên
                ans.append(i)                                   # ghi CHỈ SỐ
        return ans


sol = Solution()
for mountain, expected in examples:
    print(mountain, "->", sol.findPeaks(mountain))

[2, 4, 4] -> []
[1, 4, 3, 8, 5] -> [1, 3]
[1, 2, 3, 4, 5] -> []
[5, 1, 5] -> []


## 10. Vì sao thuật toán đúng

- **Đúng phạm vi.** Đề loại phần tử đầu và cuối, nên các ứng viên là `i = 1..n - 2`. Với các `i` này, cả `i - 1` và `i + 1` đều hợp lệ, không có truy cập ngoài biên.
- **Đúng điều kiện.** "Lớn hơn hẳn các láng giềng" nghĩa là `a[i] > a[i - 1]` **và** `a[i] > a[i + 1]`, viết gọn là `a[i - 1] < a[i] > a[i + 1]`. Dấu `>` (nghiêm ngặt) khiến đỉnh phẳng như `[2, 4, 4]` bị loại, đúng như Ví dụ 1.
- **Không sót, không trùng.** Mỗi `i` được xét đúng một lần, và quyết định cho `i` chỉ phụ thuộc vào ba giá trị `a[i-1], a[i], a[i+1]`, không phụ thuộc kết quả của `i` khác (hai đỉnh không thể kề nhau vì nếu `a[i] > a[i+1]` thì `a[i+1]` không thể lớn hơn `a[i]`).
- **Thứ tự.** Duyệt `i` tăng dần nên kết quả tăng dần (đề cho phép thứ tự bất kỳ).

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `mountain`.

- **Cách 1 (hai so sánh):** Time `O(n)`, Space `O(1)` phụ.
- **Cách 2 (dây chuyền + comprehension):** Time `O(n)`, Space `O(1)` phụ.
- **Cách 3 (`zip` ba lát cắt):** Time `O(n)`, Space `O(n)` cho các lát cắt.
- **Cách 4 (dãy hiệu):** Time `O(n)`, Space `O(n)` cho mảng hiệu.
- Phải nhìn mọi phần tử nên `O(n)` là tối ưu. Ở đây `n <= 100`, tối đa 98 lần kiểm tra.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "explicit": find_peaks_explicit,
    "chain": find_peaks_chain,
    "zip": find_peaks_zip,
    "diff": find_peaks_diff,
    "submit": sol.findPeaks,
}

for mountain, expected in examples:
    for key, fn in impls.items():
        assert sorted(fn(list(mountain))) == expected, (key, mountain, fn(list(mountain)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(mountain, expected, label):
    for key, fn in impls.items():
        got = sorted(fn(list(mountain)))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 2, 1], [1], "độ dài tối thiểu 3, đỉnh ở giữa")
check_all([1, 1, 1], [], "độ dài 3, tất cả bằng nhau")
check_all([2, 1, 2], [], "hai đầu lớn nhưng không được tính; giữa là đáy")
check_all([1, 2, 3, 4, 5], [], "tăng dần: đỉnh nằm ở cuối nên bị loại")
check_all([5, 4, 3, 2, 1], [], "giảm dần: đỉnh nằm ở đầu nên bị loại")
check_all([1, 3, 3, 1], [], "đỉnh phẳng (hai số 3 bằng nhau) không tính")
check_all([1, 3, 1, 3, 1, 3, 1], [1, 3, 5], "đỉnh xen kẽ")
check_all([100, 1, 100, 1, 100], [2], "đỉnh chỉ ở giữa; 100 ở hai đầu bị loại")
check_all([1, 100] * 50, [i for i in range(1, 99) if i % 2 == 1], "n = 100, xen kẽ 1, 100")
check_all([5] * 100, [], "n = 100, toàn bằng nhau")

PASS độ dài tối thiểu 3, đỉnh ở giữa -> [1]
PASS độ dài 3, tất cả bằng nhau -> []
PASS hai đầu lớn nhưng không được tính; giữa là đáy -> []
PASS tăng dần: đỉnh nằm ở cuối nên bị loại -> []
PASS giảm dần: đỉnh nằm ở đầu nên bị loại -> []
PASS đỉnh phẳng (hai số 3 bằng nhau) không tính -> []
PASS đỉnh xen kẽ -> [1, 3, 5]
PASS đỉnh chỉ ở giữa; 100 ở hai đầu bị loại -> [2]
PASS n = 100, xen kẽ 1, 100 -> [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67, 69, 71, 73, 75, 77, 79, 81, 83, 85, 87, 89, 91, 93, 95, 97]
PASS n = 100, toàn bằng nhau -> []


### 12.2. Tính chất và phản ví dụ

In [19]:
# Hàm kiểm tra hợp lệ (đề cho phép thứ tự bất kỳ)
def is_valid_answer(a, ans):
    if len(set(ans)) != len(ans):
        return False
    if any(not isinstance(i, int) or not (1 <= i <= len(a) - 2) for i in ans):
        return False
    expected = {i for i in range(len(a)) if 0 < i < len(a) - 1 and a[i] > a[i - 1] and a[i] > a[i + 1]}
    return set(ans) == expected


assert is_valid_answer([1, 4, 3, 8, 5], [3, 1])               # thứ tự bất kỳ
assert not is_valid_answer([1, 4, 3, 8, 5], [1])              # thiếu
assert not is_valid_answer([1, 4, 3, 8, 5], [0, 1, 3])        # chỉ số đầu mảng
print("PASS hàm kiểm tra hợp lệ")

# Tính chất 1: hai đỉnh không bao giờ kề nhau
random.seed(5)
for _ in range(500):
    a = [random.randint(1, 5) for _ in range(random.randint(3, 30))]
    ans = sol.findPeaks(a)
    assert all(ans[t + 1] - ans[t] >= 2 for t in range(len(ans) - 1))
print("PASS hai đỉnh luôn cách nhau ít nhất 2 chỉ số")

# Tính chất 2: số đỉnh <= (n - 1) // 2
for _ in range(500):
    a = [random.randint(1, 100) for _ in range(random.randint(3, 100))]
    assert len(sol.findPeaks(a)) <= (len(a) - 1) // 2
print("PASS số đỉnh <= (n - 1) // 2")

# Tính chất 3: đầu vào không bị sửa
a = [1, 4, 3, 8, 5]
copy = list(a)
for key, fn in impls.items():
    fn(a)
assert a == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: duyệt cả hai đầu -> chỉ số -1 quấn vòng
def sai_quan_vong(a):
    n = len(a)
    return [i for i in range(n - 1) if a[i - 1] < a[i] > a[i + 1]]

a = [9, 1, 2, 1, 3]
print("sai_quan_vong:", sai_quan_vong(a), "| đúng:", sol.findPeaks(a))
assert sai_quan_vong(a) == [0, 2] and sol.findPeaks(a) == [2]
print("PASS phản ví dụ: i = 0 so sánh với a[-1] và nhận sai đỉnh ở đầu mảng")

# Phản ví dụ 2: dùng >= (đỉnh phẳng)
def sai_ge(a):
    return [i for i in range(1, len(a) - 1) if a[i] >= a[i - 1] and a[i] >= a[i + 1]]

assert sai_ge([2, 4, 4]) == [1] and sol.findPeaks([2, 4, 4]) == []
print("PASS phản ví dụ: >= nhận sai đỉnh phẳng ở ví dụ 1")

PASS hàm kiểm tra hợp lệ
PASS hai đỉnh luôn cách nhau ít nhất 2 chỉ số
PASS số đỉnh <= (n - 1) // 2
PASS đầu vào không bị sửa
sai_quan_vong: [0, 2] | đúng: [2]
PASS phản ví dụ: i = 0 so sánh với a[-1] và nhận sai đỉnh ở đầu mảng
PASS phản ví dụ: >= nhận sai đỉnh phẳng ở ví dụ 1


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(2951)
for trial in range(3000):
    n = random.randint(3, 100)
    hi = random.choice([2, 3, 10, 100])                    # giá trị nhỏ -> nhiều dấu bằng
    a = [random.randint(1, hi) for _ in range(n)]
    ref = find_peaks_explicit(list(a))
    for key, fn in impls.items():
        got = fn(list(a))
        assert got == ref, (key, a, got, ref)
        assert is_valid_answer(a, got)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau và hợp lệ")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau và hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt từ `0` đến hết (chỉ số `-1` quấn vòng)

In [21]:
def sai_bien(a):
    ans = []
    for i in range(len(a) - 1):                        # bắt đầu từ i = 0
        if a[i - 1] < a[i] > a[i + 1]:                 # i = 0 -> a[-1] là phần tử CUỐI
            ans.append(i)
    return ans


a = [9, 1, 2, 1, 3]
print("sai  :", sai_bien(a), "| đúng:", Solution().findPeaks(a))

sai  : [0, 2] | đúng: [2]


**Vì sao sai:** phần tử đầu không có láng giềng trái. `a[-1]` không báo lỗi mà lấy **phần tử cuối** (`3`), nên `3 < 9 > 1` cho `i = 0` là đỉnh, trái đề. Hãy bắt đầu từ `i = 1` và dừng ở `n - 2`.

### Lỗi 2: Dùng `>=` thay vì `>`

In [22]:
def sai_ge_2(a):
    return [i for i in range(1, len(a) - 1) if a[i] >= a[i - 1] and a[i] >= a[i + 1]]


print("sai  :", sai_ge_2([2, 4, 4]), "| đúng:", Solution().findPeaks([2, 4, 4]))

sai  : [1] | đúng: []


**Vì sao sai:** đề yêu cầu **lớn hơn hẳn** (strictly). `4 >= 4` đúng nhưng `4 > 4` sai, nên `[2, 4, 4]` không có đỉnh. Phải dùng `>`.

### Lỗi 3: Chỉ so sánh với một bên

In [23]:
def sai_mot_ben(a):
    return [i for i in range(1, len(a) - 1) if a[i] > a[i - 1]]      # quên láng giềng phải


print("sai  :", sai_mot_ben([1, 2, 3, 4, 3]), "| đúng:", Solution().findPeaks([1, 2, 3, 4, 3]))

sai  : [1, 2, 3] | đúng: [3]


**Vì sao sai:** đỉnh phải lớn hơn **cả hai** bên. Trên dãy tăng `1, 2, 3, 4`, mỗi phần tử lớn hơn trái nhưng không lớn hơn phải. Cần cả `a[i] > a[i - 1]` và `a[i] > a[i + 1]`.

### Lỗi 4: Dùng `or` thay vì `and`

In [24]:
def sai_or(a):
    return [i for i in range(1, len(a) - 1) if a[i] > a[i - 1] or a[i] > a[i + 1]]


print("sai  :", sai_or([1, 4, 3, 8, 5]), "| đúng:", Solution().findPeaks([1, 4, 3, 8, 5]))

sai  : [1, 3] | đúng: [1, 3]


**Vì sao sai:** `or` chỉ đòi hỏi **một** bên thấp hơn, nên cả sườn dốc (lên hay xuống) đều bị nhận là đỉnh. Cần `and`.

### Lỗi 5: Trả giá trị thay vì chỉ số

In [25]:
def sai_gia_tri(a):
    return [a[i] for i in range(1, len(a) - 1) if a[i - 1] < a[i] > a[i + 1]]


print("sai  :", sai_gia_tri([1, 4, 3, 8, 5]), "| đúng:", Solution().findPeaks([1, 4, 3, 8, 5]))

sai  : [4, 8] | đúng: [1, 3]


**Vì sao sai:** đề hỏi **chỉ số** của đỉnh (`[1, 3]`), không phải giá trị (`[4, 8]`). Cả chỉ số và giá trị đều là số nguyên nên rất dễ lẫn; hãy đọc kỹ đề xem cần "vị trí" hay "giá trị".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "so sánh một phần tử với **láng giềng trái và phải**" → cửa sổ ba phần tử, `range(1, n - 1)`.
- Công thức nhớ: **`a[i - 1] < a[i] > a[i + 1]`** cho đỉnh, **`a[i - 1] > a[i] < a[i + 1]`** cho đáy.
- Khi đề loại hai đầu thì chọn phạm vi `i` loại hai đầu luôn, **không** cần ghi `if i == 0 ... else`.
- Cẩn thận chỉ số âm: trong Python `a[-1]` hợp lệ nhưng thường **không phải điều bạn muốn**.
- "Strictly" → `>` / `<`; "không nhỏ hơn" → `>=`. Luôn kiểm tra ví dụ có phần tử bằng nhau (như `[2, 4, 4]`).
- Cùng khung cửa sổ ba phần tử dùng cho: đếm đỉnh/đáy, dãy zigzag, làm mượt trung bình ba điểm.

## 15. Biến thể của bài

1. Tìm các **đáy (valley)**: nhỏ hơn hẳn hai láng giềng.
2. **Đếm** số đỉnh thay vì liệt kê.
3. **Đỉnh phẳng**: dãy bằng nhau liên tiếp mà hai bên đều thấp hơn (ví dụ `[1, 3, 3, 2]` có "đỉnh" kéo dài).
4. Tìm đỉnh **cao nhất** (giá trị lớn nhất trong các đỉnh) và chỉ số của nó.
5. Coi hai đầu cũng có thể là đỉnh (so với láng giềng duy nhất), như bài "Find Peak Element".

Thử biến thể 1, 3 và 5:

In [26]:
def find_valleys(a):
    return [i for i in range(1, len(a) - 1) if a[i - 1] > a[i] < a[i + 1]]


def find_plateau_peaks(a):
    """Trả về (bắt đầu, kết thúc) của các đoạn bằng nhau mà hai bên đều thấp hơn hẳn."""
    res, n, i = [], len(a), 1
    while i < n - 1:
        if a[i - 1] < a[i]:
            j = i
            while j + 1 < n and a[j + 1] == a[i]:
                j += 1
            if j + 1 < n and a[j + 1] < a[i]:
                res.append((i, j))
            i = j + 1
        else:
            i += 1
    return res


def find_peaks_with_ends(a):
    n = len(a)
    res = []
    for i in range(n):
        left_ok = i == 0 or a[i] > a[i - 1]
        right_ok = i == n - 1 or a[i] > a[i + 1]
        if left_ok and right_ok:
            res.append(i)
    return res


print("1. đáy của [5, 1, 4, 2, 6]:", find_valleys([5, 1, 4, 2, 6]))                 # [1, 3]
print("3. đỉnh phẳng của [1, 3, 3, 2, 5, 5, 5, 1]:", find_plateau_peaks([1, 3, 3, 2, 5, 5, 5, 1]))   # [(1, 2), (4, 6)]
print("5. đỉnh tính cả hai đầu của [9, 1, 4, 2, 6]:", find_peaks_with_ends([9, 1, 4, 2, 6]))        # [0, 2, 4]
assert find_valleys([5, 1, 4, 2, 6]) == [1, 3]
assert find_plateau_peaks([1, 3, 3, 2, 5, 5, 5, 1]) == [(1, 2), (4, 6)]
assert find_peaks_with_ends([9, 1, 4, 2, 6]) == [0, 2, 4]

1. đáy của [5, 1, 4, 2, 6]: [1, 3]
3. đỉnh phẳng của [1, 3, 3, 2, 5, 5, 5, 1]: [(1, 2), (4, 6)]
5. đỉnh tính cả hai đầu của [9, 1, 4, 2, 6]: [0, 2, 4]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai phép so sánh tường minh | `a[i] > a[i-1]` và `a[i] > a[i+1]` | `O(n)` | `O(1)` | học định nghĩa, dễ debug |
| 2. Dây chuyền + comprehension | `a[i-1] < a[i] > a[i+1]` | `O(n)` | `O(1)` | **cách nộp bài**, gọn mà rõ |
| 3. `zip` ba lát cắt | bộ `(trái, giữa, phải)` | `O(n)` | `O(n)` | không muốn viết `i ± 1` |
| 4. Dãy hiệu | dấu `+` rồi `-` | `O(n)` | `O(n)` | mở rộng cho đỉnh phẳng / dãy zigzag |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(range(1, 6 - 1))`, `3 < 5 > 4`, `2 < 2 > 1`, `Solution().findPeaks([1, 3, 2])`, `Solution().findPeaks([3, 2, 3])`.

**Bài 2 (tính tay).** Với `mountain = [2, 7, 3, 9, 9, 4, 8, 1]`: lập bảng `(trái, giữa, phải)` cho từng `i` và cho biết các đỉnh. Chú ý hai số `9` kề nhau.

**Bài 3 (điền TODO).** Hoàn thành `find_peaks_todo(mountain)`.

In [27]:
def find_peaks_todo(mountain):
    ans = []
    n = len(mountain)
    # TODO 1: duyệt i từ 1 đến n - 2 (dùng range(1, n - 1))
    #     TODO 2: nếu mountain[i - 1] < mountain[i] > mountain[i + 1] thì ans.append(i)
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_peaks_diff2(mountain)` bằng **dãy hiệu** (Cách 4) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_peaks_and_valleys(a)` trả về cặp `(số đỉnh, số đáy)`. (b) Viết `highest_peak(a)` trả về `(chỉ số, giá trị)` của đỉnh cao nhất (hòa thì lấy chỉ số nhỏ hơn), hoặc `None` nếu không có đỉnh. (c) Viết `is_mountain_shape(a)` trả `True` nếu mảng có đúng **một** đỉnh và tăng nghiêm ngặt trước đỉnh, giảm nghiêm ngặt sau đỉnh (như bài "Valid Mountain Array").

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 2, 3, 4]`, `True`, `False`, `[1]`, `[]`. Bài 3: vòng `for i in range(1, n - 1):` rồi `if ...: ans.append(i)`.
- **Mức 2 (vừa):** Bài 2: `i=1`: `(2, 7, 3)` đỉnh; `i=2`: `(7, 3, 9)` không; `i=3`: `(3, 9, 9)` không (`9 > 9` sai); `i=4`: `(9, 9, 4)` không; `i=5`: `(9, 4, 8)` không; `i=6`: `(4, 8, 1)` đỉnh. Kết quả `[1, 6]`. Bài 4: `d = [a[t+1]-a[t] ...]`, đỉnh khi `d[t-1] > 0 and d[t] < 0`.
- **Mức 3 (sâu):** Bài 5(a): đáy dùng `a[i-1] > a[i] < a[i+1]`. 5(b): lấy danh sách đỉnh rồi `max(peaks, key=lambda i: (a[i], -i))`. 5(c): có đúng một đỉnh, các hiệu trước đỉnh đều dương, sau đỉnh đều âm; đỉnh không thể ở hai đầu (`0 < p < n - 1`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert list(range(1, 6 - 1)) == [1, 2, 3, 4]
assert (3 < 5 > 4) is True and (2 < 2 > 1) is False
assert Solution().findPeaks([1, 3, 2]) == [1] and Solution().findPeaks([3, 2, 3]) == []

# Bài 2
assert Solution().findPeaks([2, 7, 3, 9, 9, 4, 8, 1]) == [1, 6]

# Bài 3
def find_peaks_todo_ans(mountain):
    ans = []
    n = len(mountain)
    for i in range(1, n - 1):
        if mountain[i - 1] < mountain[i] > mountain[i + 1]:
            ans.append(i)
    return ans

for mountain, expected in examples:
    assert find_peaks_todo_ans(mountain) == expected

# Bài 4
def find_peaks_diff2(mountain):
    d = [mountain[t + 1] - mountain[t] for t in range(len(mountain) - 1)]
    return [t for t in range(1, len(mountain) - 1) if d[t - 1] > 0 and d[t] < 0]

for mountain, expected in examples:
    assert find_peaks_diff2(mountain) == expected

# Bài 5(a)
def count_peaks_and_valleys(a):
    peaks = sum(1 for i in range(1, len(a) - 1) if a[i - 1] < a[i] > a[i + 1])
    valleys = sum(1 for i in range(1, len(a) - 1) if a[i - 1] > a[i] < a[i + 1])
    return peaks, valleys

assert count_peaks_and_valleys([1, 4, 3, 8, 5]) == (2, 1)
assert count_peaks_and_valleys([1, 2, 3]) == (0, 0)

# Bài 5(b)
def highest_peak(a):
    peaks = Solution().findPeaks(a)
    if not peaks:
        return None
    best = max(peaks, key=lambda i: (a[i], -i))
    return best, a[best]

assert highest_peak([1, 4, 3, 8, 5]) == (3, 8)
assert highest_peak([1, 7, 3, 7, 1]) == (1, 7)          # hòa: chỉ số nhỏ hơn
assert highest_peak([1, 2, 3]) is None

# Bài 5(c)
def is_mountain_shape(a):
    peaks = Solution().findPeaks(a)
    if len(peaks) != 1:
        return False
    p = peaks[0]
    return all(a[t] < a[t + 1] for t in range(p)) and all(a[t] > a[t + 1] for t in range(p, len(a) - 1))

assert is_mountain_shape([1, 3, 5, 4, 2]) is True
assert is_mountain_shape([1, 3, 3, 2]) is False
assert is_mountain_shape([1, 3, 2, 4, 1]) is False
assert is_mountain_shape([1, 2, 3]) is False
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ ba phần tử, bỏ hai đầu)

```python
from typing import List

class Solution:
    def findPeaks(self, mountain: List[int]) -> List[int]:
        ans = []
        for i in range(1, len(mountain) - 1):
            if mountain[i - 1] < mountain[i] > mountain[i + 1]:
                ans.append(i)
        return ans
```

**Checklist**

- [ ] Vòng lặp chạy `i` từ `1` đến `n - 2` (`range(1, n - 1)`), loại phần tử đầu và cuối?
- [ ] Điều kiện lớn hơn **hẳn** (`>`), không phải `>=`?
- [ ] Đã so sánh với **cả hai** láng giềng bằng `and` (hoặc dây chuyền)?
- [ ] Trả về **chỉ số** `i`, không phải giá trị `mountain[i]`?
- [ ] Không có chỗ nào truy cập `a[-1]` hay `a[n]` ngoài ý muốn?
- [ ] Đã thử: `[2, 4, 4]` (đỉnh phẳng), mảng tăng/giảm dần, `n = 3`, mảng toàn bằng nhau?
- [ ] Skeleton đúng `class Solution` / `def findPeaks(self, mountain: List[int]) -> List[int]`?